#  Download Audio Dataset

**Notebook name:** `01_data_download.ipynb`  
**Location in Drive:** `CSE499_EHR_Project/02_Phase1_ASR/notebooks/`

**Purpose:** Download Bangla multi-dialect audio from YouTube and other sources, rename files following the naming convention `[dialect_code]_[3-digit number]_[gender]_[age group].mp3`, organize into dialect folders, and update `dataset_log.csv`.

**What gets saved:**
- MP3 audio files → `CSE499_EHR_Project/01_Dataset/raw_audio/[dialect]/`
- Updated `dataset_log.csv` → `CSE499_EHR_Project/01_Dataset/metadata/`


## Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/CSE499_EHR_Project'
assert os.path.exists(ROOT), f'ERROR: {ROOT} not found. Run 00_project_setup.ipynb first.'
print(f'✅ Drive mounted. Project root: {ROOT}')

Mounted at /content/drive
✅ Drive mounted. Project root: /content/drive/MyDrive/CSE499_EHR_Project


## Install yt-dlp


In [2]:
!pip install -q yt-dlp
print('✅ yt-dlp installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.2 MB/s eta 0:00:00
✅ yt-dlp installed


## Define paths and dialect codes
These are the exact dialect folder names and codes.The naming convention is: `[dialect_code]_[3-digit number]_[gender]_[age group].mp3`

| Dialect Code | Dialect | Folder Name |
|---|---|---|
| pd | Puran Dhaka (Old Dhaka) | puran_dhaka |
| br | Barishal | barishal |
| sy | Sylheti | sylheti |
| nb | Normal Bangla | normal_bangla |
| ib | Indian Bangla | indian_bangla |

In [3]:
import csv
import shutil
import subprocess

RAW_AUDIO_DIR = f'{ROOT}/01_Dataset/raw_audio'
METADATA_DIR = f'{ROOT}/01_Dataset/metadata'

DIALECT_MAP = {
    'pd': 'puran_dhaka',
    'br': 'barishal',
    'sy': 'sylheti',
    'nb': 'normal_bangla',
    'ib': 'indian_bangla',
}

# Verify all dialect folders exist
for code, folder in DIALECT_MAP.items():
    path = os.path.join(RAW_AUDIO_DIR, folder)
    assert os.path.exists(path), f'Missing folder: {path}'
    print(f'✅ {code} → {folder}/ exists')


def count_existing_files(dialect_code):
    """Count how many audio files already exist for a dialect."""
    folder = os.path.join(RAW_AUDIO_DIR, DIALECT_MAP[dialect_code])
    files = [f for f in os.listdir(folder)
             if f.endswith(('.mp3', '.wav', '.m4a', '.flac', '.ogg'))]
    return len(files), sorted(files)


def get_next_numbers():
    """Get the next available file number for each dialect."""
    next_num = {}
    for code in DIALECT_MAP:
        count, _ = count_existing_files(code)
        next_num[code] = count + 1
    return next_num


def get_duration(filepath):
    """Get audio duration in seconds using ffprobe."""
    try:
        cmd = ['ffprobe', '-v', 'error', '-show_entries',
               'format=duration', '-of', 'csv=p=0', filepath]
        result = subprocess.run(cmd, capture_output=True, text=True)
        return round(float(result.stdout.strip()), 1)
    except:
        return 'unknown'


def log_to_csv(filename, source, dialect, gender, age_group, duration):
    """Append one row to dataset_log.csv."""
    csv_path = f'{METADATA_DIR}/dataset_log.csv'
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([filename, source, dialect, gender, age_group, duration])


print(f'\n📂 Raw audio root: {RAW_AUDIO_DIR}')

✅ pd → puran_dhaka/ exists
✅ br → barishal/ exists
✅ sy → sylheti/ exists
✅ nb → normal_bangla/ exists
✅ ib → indian_bangla/ exists

📂 Raw audio root: /content/drive/MyDrive/CSE499_EHR_Project/01_Dataset/raw_audio


## Count existing files per dialect

In [4]:
print('Existing audio files per dialect:')
print('-' * 40)
next_number = get_next_numbers()
for code, folder_name in DIALECT_MAP.items():
    count, files = count_existing_files(code)
    print(f'  {code} ({folder_name}): {count} files')

print(f'\nNext file number per dialect: {next_number}')

Existing audio files per dialect:
----------------------------------------
  pd (puran_dhaka): 1 files
  br (barishal): 2 files
  sy (sylheti): 2 files
  nb (normal_bangla): 3 files
  ib (indian_bangla): 2 files

Next file number per dialect: {'pd': 2, 'br': 3, 'sy': 3, 'nb': 4, 'ib': 3}


### Upload pre-named audio files

In [5]:
import pandas as pd

csv_path = f'{METADATA_DIR}/dataset_log.csv'

# Load existing CSV entries to know which files are already registered
existing_entries = set()
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    if 'filename' in df.columns:
        existing_entries = set(df['filename'].tolist())

new_count = 0

for code, folder_name in DIALECT_MAP.items():
    folder_path = os.path.join(RAW_AUDIO_DIR, folder_name)
    audio_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(('.mp3', '.wav', '.m4a', '.flac', '.ogg'))
    ]

    for filename in sorted(audio_files):
        if filename in existing_entries:
            continue

        filepath = os.path.join(folder_path, filename)
        duration = get_duration(filepath)

        # Parse metadata from filename
        parts = os.path.splitext(filename)[0].split('_')
        gender = parts[2] if len(parts) > 2 else 'unknown'
        age_group = parts[3] if len(parts) > 3 else 'unknown'

        log_to_csv(filename, 'drive_upload', code, gender, age_group, duration)
        new_count += 1
        print(f'  ✅ Registered: {filename} ({duration}s) in {folder_name}/')

if new_count == 0:
    print('ℹ️  No new unregistered files found. All files in Drive are already in dataset_log.csv.')
else:
    print(f'\n✅ Registered {new_count} new files in dataset_log.csv')

  ✅ Registered: pd_001_male_40s.mp3 (2144.4s) in puran_dhaka/
  ✅ Registered: br_001_male_25s.mp3 (1402.3s) in barishal/
  ✅ Registered: br_002_female_30s.mp3 (625.8s) in barishal/
  ✅ Registered: sy_001_male_26s.mp3 (738.6s) in sylheti/
  ✅ Registered: sy_002_male_26s.mp3 (1163.1s) in sylheti/
  ✅ Registered: nb_001_male_30s.mp3 (300.0s) in normal_bangla/
  ✅ Registered: nb_002_male_50s.mp3 (250.3s) in normal_bangla/
  ✅ Registered: nb_003_female_40s.mp3 (4668.4s) in normal_bangla/
  ✅ Registered: ib_001_male_50s.mp3 (3785.9s) in indian_bangla/
  ✅ Registered: ib_002_male_50s.mp3 (1965.5s) in indian_bangla/

✅ Registered 10 new files in dataset_log.csv


## Verify all downloaded files and dataset_log.csv

In [6]:
import pandas as pd

print('=== Audio files per dialect ===')
total = 0
for code, folder_name in DIALECT_MAP.items():
    count, file_list = count_existing_files(code)
    total += count
    print(f'  {code} ({folder_name}): {count} files')
    if count > 0:
        print(f'     First: {file_list[0]}')
        print(f'     Last:  {file_list[-1]}')
print(f'  TOTAL: {total} files\n')

# Preview the dataset log
csv_path = f'{METADATA_DIR}/dataset_log.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'=== dataset_log.csv: {len(df)} entries ===')
    if len(df) > 0:
        print(df.head(10).to_string(index=False))
        print(f'\nSource breakdown:')
        if 'source_url' in df.columns:
            col = 'source_url'
        else:
            col = df.columns[1]  # Second column is typically source
        print(df[col].value_counts().to_string())
    else:
        print('(empty)')
else:
    print('⚠️  dataset_log.csv not found. Run 00_project_setup.ipynb first.')

=== Audio files per dialect ===
  pd (puran_dhaka): 1 files
     First: pd_001_male_40s.mp3
     Last:  pd_001_male_40s.mp3
  br (barishal): 2 files
     First: br_001_male_25s.mp3
     Last:  br_002_female_30s.mp3
  sy (sylheti): 2 files
     First: sy_001_male_26s.mp3
     Last:  sy_002_male_26s.mp3
  nb (normal_bangla): 3 files
     First: nb_001_male_30s.mp3
     Last:  nb_003_female_40s.mp3
  ib (indian_bangla): 2 files
     First: ib_001_male_50s.mp3
     Last:  ib_002_male_50s.mp3
  TOTAL: 10 files

=== dataset_log.csv: 10 entries ===
             filename   source_url dialect speaker_gender approximate_age  duration_seconds
  pd_001_male_40s.mp3 drive_upload      pd           male             40s            2144.4
  br_001_male_25s.mp3 drive_upload      br           male             25s            1402.3
br_002_female_30s.mp3 drive_upload      br         female             30s             625.8
  sy_001_male_26s.mp3 drive_upload      sy           male             26s           

---
## ✅ Part Complete

**What was saved to Google Drive:**
- Audio files in `01_Dataset/raw_audio/[dialect]/` following naming convention
- `dataset_log.csv` updated with file metadata
